In [1]:
%pip install -qU \
  langchain \
  langchain-core \
  langchain-groq \
  langchain-text-splitters \
  langchain-huggingface \
  langchain-community \
  faiss-cpu \
  sentence-transformers \
  pypdf \
  gradio

In [2]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    groq_key = userdata.get("GROQ_API_KEY")
except Exception:
    groq_key = None

if not groq_key:
    groq_key = getpass("Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = groq_key
os.environ["GROQ_MODEL"] = "openai/gpt-oss-120b"
os.environ["EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"

print("Groq key loaded:", "Yes" if os.environ.get("GROQ_API_KEY") else "No")
print("Groq model:", os.environ["GROQ_MODEL"])
print("Embedding model:", os.environ["EMBEDDING_MODEL"])

Groq key loaded: Yes
Groq model: openai/gpt-oss-120b
Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [3]:
from pathlib import Path
import xml.etree.ElementTree as ET

from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

DATA_DIR = Path("/content/data/raw")
VECTOR_DB_PATH = Path("/content/faiss_index")

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully.")
print("Data folder:", DATA_DIR)

/tmp/ipykernel_3828/3898541187.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Libraries imported successfully.
Data folder: /content/data/raw


In [4]:
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    file_path = DATA_DIR / filename
    file_path.write_bytes(content)
    print("Uploaded:", file_path)

print("\nCurrent files:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Saving AI For Everyone Fundamentals.pdf to AI For Everyone Fundamentals (1).pdf
Uploaded: /content/data/raw/AI For Everyone Fundamentals (1).pdf

Current files:
- AI For Everyone Fundamentals.pdf
- AI For Everyone Fundamentals (1).pdf
- 2026 PROGRESS REPORT.docx


In [5]:
def read_txt_or_md(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf(path: Path):
    reader = PdfReader(str(path))
    docs = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "page": page_number,
                        "type": "pdf"
                    }
                )
            )

    return docs


def read_xml(path: Path) -> str:
    try:
        tree = ET.parse(path)
        root = tree.getroot()
        text_parts = []

        for element in root.iter():
            if element.text and element.text.strip():
                text_parts.append(element.text.strip())

        return "\n".join(text_parts)
    except Exception:
        return path.read_text(encoding="utf-8", errors="ignore")


def load_documents_from_directory(directory=DATA_DIR):
    documents = []
    supported_extensions = {".txt", ".md", ".pdf", ".xml"}

    for path in sorted(Path(directory).rglob("*")):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix not in supported_extensions:
            continue

        if suffix == ".pdf":
            documents.extend(read_pdf(path))

        elif suffix in {".txt", ".md"}:
            text = read_txt_or_md(path)
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "type": suffix.replace(".", "")
                    }
                )
            )

        elif suffix == ".xml":
            text = read_xml(path)
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "type": "xml"
                    }
                )
            )

    return documents


documents = load_documents_from_directory(DATA_DIR)

print("STEP 1: DATA INGESTION")
print("=" * 50)
print("Documents loaded:", len(documents))

for i, doc in enumerate(documents[:3], start=1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print("Source:", doc.metadata.get("source"))
    print("Type:", doc.metadata.get("type"))
    print(doc.page_content[:500])

STEP 1: DATA INGESTION
Documents loaded: 18

Document 1
--------------------------------------------------
Source: AI For Everyone Fundamentals (1).pdf
Type: pdf
AI for Everyone: Fundamentals 
ISBN: 978-81-957387-3-1  99 
 
Chapter 10: Introduction to Artificial Intelligence (AI) 
 
Dr . Kurundkar Gajanan D. 
Assistant Professor, Department of Computer Science, 
Shri Guru Buddhiswami Mahavidyalaya, Purna (Jn.) Dist. Parbhani 
Mail ID: gajanan.kurundkar@gmail.com 
 
Definition of AI: 
Artificial Intelligence refers to the development of intelligent machines that can perform tasks 
that typically require human intelligence. It is a branch of computer scien

Document 2
--------------------------------------------------
Source: AI For Everyone Fundamentals (1).pdf
Type: pdf
AI for Everyone: Fundamentals 
ISBN: 978-81-957387-3-1  100 
 
While we have made significant progress in narrow AI, achieving true General AI is still a subject 
of ongoing research and development. Artificial general 

In [6]:
def split_documents(documents, chunk_size=800, chunk_overlap=120):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    return splitter.split_documents(documents)


chunks = split_documents(documents, chunk_size=800, chunk_overlap=120)

print("STEP 2: DATA TRANSFORMATION")
print("=" * 50)
print("Original documents:", len(documents))
print("Chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\nChunk {i}")
    print("-" * 50)
    print("Source:", chunk.metadata.get("source"))
    print(chunk.page_content[:500])

STEP 2: DATA TRANSFORMATION
Original documents: 18
Chunks created: 70

Chunk 1
--------------------------------------------------
Source: AI For Everyone Fundamentals (1).pdf
AI for Everyone: Fundamentals 
ISBN: 978-81-957387-3-1  99 
 
Chapter 10: Introduction to Artificial Intelligence (AI) 
 
Dr . Kurundkar Gajanan D. 
Assistant Professor, Department of Computer Science, 
Shri Guru Buddhiswami Mahavidyalaya, Purna (Jn.) Dist. Parbhani 
Mail ID: gajanan.kurundkar@gmail.com 
 
Definition of AI: 
Artificial Intelligence refers to the development of intelligent machines that can perform tasks 
that typically require human intelligence. It is a branch of computer scien

Chunk 2
--------------------------------------------------
Source: AI For Everyone Fundamentals (1).pdf
AI systems are designed to analyse and interpret large amounts of data, recognize patterns and 
correlations, and make predictions or take actions based on that information. They often employ 
algorithms and models to p

In [7]:
EMBEDDING_MODEL = os.environ.get(
    "EMBEDDING_MODEL",
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(str(VECTOR_DB_PATH))

print("STEP 3: EMBEDDINGS + FAISS")
print("=" * 50)
print("Embedding model:", EMBEDDING_MODEL)
print("Chunks embedded:", len(chunks))
print("Vector database saved at:", VECTOR_DB_PATH)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

STEP 3: EMBEDDINGS + FAISS
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Chunks embedded: 70
Vector database saved at: /content/faiss_index


In [8]:
question = "What is an AI?"

results = vector_store.similarity_search(question, k=3)

print("STEP 4: VECTOR SEARCH")
print("=" * 50)
print("Question:", question)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("-" * 50)
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])

STEP 4: VECTOR SEARCH
Question: What is an AI?

Result 1
--------------------------------------------------
Source: AI For Everyone Fundamentals (1).pdf
Intelligence.[1]. Artificial Intel ligence (AI) refers to the simulation of human intelligence in 
machines that are programmed to perform tasks that would typically require human intelligence. 
AI enables machines to analyse, interpret, and understand information, make decisions, and learn 
from experiences. 
 
There are two primary types of AI:  
 
Narrow AI and 2. General AI: 
Narrow AI: Narrow AI, also known as Weak AI, is designed to perform specific tasks within a 
defined scope. Examples of narrow AI include voice assistants like Siri and Alexa, image 
recognition software, and recommendation algorithms used by online platforms. 
 
General AI: General AI, also known as Strong AI, aims

Result 2
--------------------------------------------------
Source: AI For Everyone Fundamentals.pdf
Intelligence.[1]. Artificial Intel ligence (

In [9]:
def get_llm(temperature=0.1):
    return ChatGroq(
        model=os.environ.get("GROQ_MODEL", "qwen/qwen3.6-27b"),
        temperature=temperature,
        api_key=os.environ.get("GROQ_API_KEY")
    )


def answer_with_groq(question, k=3, temperature=0.1):
    retrieved_docs = vector_store.similarity_search(question, k=k)

    context = "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}] {doc.page_content}"
        for doc in retrieved_docs
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful PGD Generative AI teaching assistant. "
                "Answer using only the given context. "
                "Use simple classroom language. "
                "If the answer is not in the context, say: "
                "'I do not know from the uploaded documents.'"
            ),
            (
                "human",
                "Context:\n{context}\n\nQuestion:\n{question}\n\n"
                "Give a clear answer."
            )
        ]
    )

    llm = get_llm(temperature=temperature)
    chain = prompt | llm | StrOutputParser()

    answer = chain.invoke(
        {
            "context": context,
            "question": question
        }
    )

    sources = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page")
        label = f"{source}, page {page}" if page else source
        if label not in sources:
            sources.append(label)

    return answer, sources


answer, sources = answer_with_groq("What are benefits of AI?", k=3)

print("Groq Answer")
print("=" * 50)
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Groq Answer
**Benefits of AI (as described in the provided material)**  

1. **Helps people focus on higher‑level work** – AI takes care of repetitive and mundane tasks, so humans can spend more time on activities that need creativity, critical thinking, and empathy.  

2. **Augments human abilities** – By working together with AI, we can do things faster and more accurately than we could on our own.  

3. **Boosts productivity** – Automating routine work frees up time and resources, allowing individuals and organizations to get more done.  

4. **Supports complex problem‑solving** – With AI handling the “busy work,” people have more mental bandwidth to tackle complex challenges and innovate.  

5. **Drives future transformation** – The future of AI promises continued advances that can reshape many fields (healthcare, education, industry, etc.), creating new opportunities and improvements.  

These points capture the main advantages of AI mentioned in the uploaded documents.

Sources:


In [10]:
my_question = "Is there anything about RAG pipeline in the document?"

answer, sources = answer_with_groq(my_question, k=4, temperature=0.1)

print("Question:", my_question)
print("\nAnswer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Question: Is there anything about RAG pipeline in the document?

Answer:
I do not know from the uploaded documents.

Sources:
- AI For Everyone Fundamentals (1).pdf, page 7
- AI For Everyone Fundamentals.pdf, page 7


In [20]:
import os
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

# ==========================================
# BACKEND FUNCTIONS
# ==========================================

def simple_groq_chat(message, temperature):
    """
    Handles direct Groq LLM queries for the Chat Playground tab.
    """
    try:
        # Initialize the Groq chat model
        llm = ChatGroq(
            model="openai/gpt-oss-120b",
            temperature=float(temperature)
        )

        # Invoke the model with the user prompt
        response = llm.invoke([HumanMessage(content=message)])
        return response.content

    except Exception as e:
        return f"Error executing Groq chat: {str(e)}"


def rag_ui_answer(question, k, temperature):
    if not question or not question.strip():
        return "Please enter a question."

    try:
        # Ensure answer_with_groq is defined elsewhere in your notebook for RAG
        answer, sources = answer_with_groq(
            question=question,
            k=int(k),
            temperature=float(temperature)
        )

        source_text = "\n".join(f"- {source}" for source in sources)

        return f"""### ⚡ Answer & Insights
{answer}

---
### 📁 Retrieved Sources
{source_text}
"""

    except Exception as e:
        return f"""### ⚠️ Error Encountered
`{str(e)}`

**Please verify:**
1. Groq API key environment variable is set (`os.environ["GROQ_API_KEY"]`)
2. Documents/PDFs are successfully loaded into FAISS
3. Model parameters are configured properly
"""


def playground_ui(message, temperature):
    if not message or not message.strip():
        return "Please enter a prompt."

    try:
        return simple_groq_chat(message, temperature=float(temperature))
    except Exception as e:
        return f"Error: {str(e)}"


# ==========================================
# CUSTOM STYLING (CSS)
# ==========================================

CUSTOM_CSS = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
}
#hero {
    padding: 32px;
    border-radius: 24px;
    background: linear-gradient(135deg, #18080a, #2b0b10, #3d1217);
    color: white;
    box-shadow: 0 20px 40px rgba(220, 38, 38, 0.15);
    margin-bottom: 24px;
    border: 1px solid rgba(239, 68, 68, 0.2);
}
#hero h1 {
    font-size: 40px;
    font-weight: 800;
    margin-bottom: 10px;
    background: linear-gradient(90deg, #f87171, #fb923c, #f43f5e);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
#hero p {
    color: #fdba74;
    font-size: 16px;
    line-height: 1.5;
}
.gr-button.gr-button-primary {
    background: linear-gradient(135deg, #ef4444, #f97316) !important;
    border: none !important;
    font-weight: 600 !important;
    color: white !important;
    transition: all 0.3s ease !important;
}
.gr-button.gr-button-primary:hover {
    background: linear-gradient(135deg, #dc2626, #ea580c) !important;
    box-shadow: 0 4px 15px rgba(239, 68, 68, 0.4) !important;
}
.tabs {
    border-color: #fca5a5 !important;
}
"""


# ==========================================
# GRADIO INTERFACE (UI)
# ==========================================

with gr.Blocks(css=CUSTOM_CSS, title="Groq RAG Studio", theme=gr.themes.Soft(primary_hue="rose", secondary_hue="orange")) as demo:
    gr.HTML(
        """
        <div id="hero">
            <h1>⚡ Groq RAG Studio</h1>
            <p>Advanced Document Intelligence using LangChain, FAISS Vector Search, and Groq LLM API.</p>
            <p>Query secure notes, PDFs, TXT, MD, or XML files safely.</p>
        </div>
        """
    )

    with gr.Tabs():
        with gr.Tab("📚 Ask Your Documents"):
            gr.Markdown("### Document Query Engine\nAsk questions based on your ingested vector records.")

            with gr.Row():
                with gr.Column(scale=3):
                    question_box = gr.Textbox(
                        label="Your Question",
                        placeholder="Example: What is RAG? What is LangChain?",
                        lines=3
                    )
                    with gr.Row():
                        k_slider = gr.Slider(1, 8, value=4, step=1, label="Retrieved Chunks (k)")
                        temp_slider = gr.Slider(0, 1, value=0.1, step=0.1, label="Temperature")

                    ask_btn = gr.Button("Ask Groq", variant="primary", size="lg")

            output_box = gr.Markdown(label="Response Summary", value="")

            ask_btn.click(
                fn=rag_ui_answer,
                inputs=[question_box, k_slider, temp_slider],
                outputs=output_box
            )

        with gr.Tab("🧠 Groq Chat Playground"):
            gr.Markdown("### Direct LLM Playground\nTest queries straight against the Groq model without vector database indexing.")

            prompt_box = gr.Textbox(
                label="Prompt",
                placeholder="Explain prompt engineering in simple words.",
                lines=4
            )
            playground_temp = gr.Slider(0, 1, value=0.4, step=0.1, label="Temperature")
            generate_btn = gr.Button("Generate with Groq", variant="primary")
            playground_output = gr.Textbox(label="Groq Output", lines=10)

            generate_btn.click(
                fn=playground_ui,
                inputs=[prompt_box, playground_temp],
                outputs=playground_output
            )

        with gr.Tab("🧩 RAG Pipeline Explanation"):
            gr.Markdown(
                """
                ## 🔄 End-to-End RAG Architecture
                1. **Data Ingestion**: Parses documents (PDF, TXT, MD, XML).
                2. **Text Chunking**: Breaks extensive files down into structured text slices.
                3. **Vector Embeddings**: Converts text into numerical high-dimensional matrices.
                4. **FAISS Indexing**: Local indexing for high-speed record matching.
                5. **Context Retrieval**: Pulls the top-$k$ most relevant document fragments.
                6. **Groq Inference**: Generates evidence-grounded answers.
                7. **Gradio Interface**: Renders a clean, accessible layout.
                """
            )

demo.launch(share=True, debug=True)

/tmp/ipykernel_3828/3753777795.py:125: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Groq RAG Studio", theme=gr.themes.Soft(primary_hue="rose", secondary_hue="orange")) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://212bf0b804a94ba564.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://212bf0b804a94ba564.gradio.live
